**데이터 전처리(Data Preprocessing) 개념 정리**

**1. 데이터 전처리 정의 및 목적**

- **정의**: 원시 데이터(raw data)를 분석 및 머신러닝에 적합한 형태 변환
- **목적**: 데이터 품질 향상, 분석/모델링 과정 오류 최소화 및 정확한 결과 도출

**2. 데이터 전처리  작업**

- **정제(Cleansing)**
    - **결측치 처리**:
        - 개념: 누락된 값(Missing values) 식별 및 삭제/대체
        - *식별*: 단순 탐색, 시각화, 결측치 패턴 분석
        - *처리*: 삭제, 평균/중앙값/최빈값 대체, 전진/후진 채우기, 임의 값, KNN/회귀/다중 대체
    - **이상치 처리**:
        - 개념 :  다른 데이터 포인트와 현저히 차이 나는 값 처리
        - *식별*: IQR 방법, 박스플롯(Boxplot) 사용, 상한선/하한선 설정
        - *처리*: 이상치 제거, 평균/중앙값/최빈값 대체, 로그/제곱근 변환, 클리핑(Clipping), 회귀 모델/KNN 대체
    - **중복 데이터 제거**: 동일 데이터 레코드 정리
- **통합(Integration)**:
    - 개념: 분석을 위한 여러 데이터원 통합
- **변환(Transformation)**
    - 데이터 분포 정상화, 범위 표준화, 이상치 영향 감소 등을 목적으로 다루기 쉬운 형식 변환
    - 척도 맞추기 작업, 스케일링, 정규화, 범주형 데이터 인코딩
- **데이터 축소(Reduction)**: 차원(변수) 축소, 샘플링
- **특징 선택 및 생성(Feature Selection & Engineering)**: 모델 성능 향상을 위한 주요 특징(변수) 선택 및 생성

Seaborn의 `titanic` 데이터셋과 Scikit-learn 라이브러리를 활용해 전처리 전 과정을 구현한 예시 코드입니다.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 데이터 로드
df = sns.load_dataset("titanic")

# ==========================================
# 1. 기초 정제 (Cleansing & Feature)
# ==========================================
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

q1 = df["fare"].quantile(0.25)
q3 = df["fare"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
lower_bound = q1 - 1.5 * iqr
df["fare"] = np.clip(df["fare"], lower_bound, upper_bound)

df["family_size"] = df["sibsp"] + df["parch"] + 1
df = df.drop_duplicates().reset_index(drop=True)


# ==========================================
# 2. 원본 데이터 상태에서 먼저 분할 (Data Leakage 방지)
# ==========================================
X_raw = df[["sex", "age", "fare", "family_size"]]
y = df["survived"]

# 1차 분할: Train+Val (80%) / Test (20%)
X_train_val_raw, X_test_raw, y_train_val, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# 2차 분할: Train (60%) / Validation (20%)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_val_raw, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)


# ==========================================
# 3. Train 데이터 기준으로만 fit 수행 및 각각 transform
# ==========================================

# 3-1. OneHotEncoder (fit: Train 전용)
encoder = OneHotEncoder(sparse_output=False, drop="first")
sex_train_enc = pd.DataFrame(encoder.fit_transform(X_train_raw[["sex"]]), columns=encoder.get_feature_names_out(["sex"]), index=X_train_raw.index)
sex_val_enc = pd.DataFrame(encoder.transform(X_val_raw[["sex"]]), columns=encoder.get_feature_names_out(["sex"]), index=X_val_raw.index)

# 3-2. StandardScaler (fit: Train 전용)
scaler = StandardScaler()
num_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_raw[["age", "fare"]]), columns=["age_scaled", "fare_scaled"], index=X_train_raw.index)
num_val_scaled = pd.DataFrame(scaler.transform(X_val_raw[["age", "fare"]]), columns=["age_scaled", "fare_scaled"], index=X_val_raw.index)

# 전처리 결과 결합
X_train = pd.concat([num_train_scaled, X_train_raw[["family_size"]], sex_train_enc], axis=1)
X_val = pd.concat([num_val_scaled, X_val_raw[["family_size"]], sex_val_enc], axis=1)

# 3-3. PCA (fit: Train 전용)
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)


# ==========================================
# 4. 모델 학습 & Validation 테스트
# ==========================================
model = LogisticRegression(solver="saga", l1_ratio=0.0, random_state=42)
model.fit(X_train_pca, y_train)

y_val_pred = model.predict(X_val_pca)
print("--- [Validation] Performance ---")
print(classification_report(y_val, y_val_pred))


# ==========================================
# 5. 전처리기(Scaler, Encoder, PCA) & 모델 한번에 저장
# ==========================================
save_dir = "model"
os.makedirs(save_dir, exist_ok=True)

# 저장할 객체들을 딕셔너리로 패키징
artifacts = {
    "encoder": encoder,
    "scaler": scaler,
    "pca": pca,
    "model": model
}

artifacts_path = os.path.join(save_dir, "titanic_pipeline.pkl")
joblib.dump(artifacts, artifacts_path)
print(f"\n전처리기 및 모델 저장 완료: {artifacts_path}")


# ==========================================
# 6. 저장된 아티팩트 불러오기 후 Test 데이터에 적용 및 실행
# ==========================================

# 1) 저장된 아티팩트 로드
loaded_artifacts = joblib.load(artifacts_path)

loaded_encoder = loaded_artifacts["encoder"]
loaded_scaler = loaded_artifacts["scaler"]
loaded_pca = loaded_artifacts["pca"]
loaded_model = loaded_artifacts["model"]

# 2) 불러온 Scaler 및 Encoder로 원본 Test 데이터(X_test_raw) 변환
test_sex_enc = pd.DataFrame(
    loaded_encoder.transform(X_test_raw[["sex"]]),
    columns=loaded_encoder.get_feature_names_out(["sex"]),
    index=X_test_raw.index
)
test_num_scaled = pd.DataFrame(
    loaded_scaler.transform(X_test_raw[["age", "fare"]]),
    columns=["age_scaled", "fare_scaled"],
    index=X_test_raw.index
)
X_test_prepared = pd.concat([test_num_scaled, X_test_raw[["family_size"]], test_sex_enc], axis=1)

# 3) 불러온 PCA 적용
X_test_pca = loaded_pca.transform(X_test_prepared)

# 4) 불러온 모델로 Test 예측
y_test_pred = loaded_model.predict(X_test_pca)

print("\n--- [Loaded Model - Test Data] Performance ---")
print(classification_report(y_test, y_test_pred))